[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/56_adamw.ipynb)

# 🔴 Hard: AdamW (Decoupled Weight Decay)

Implement **AdamW** (Loshchilov & Hutter 2019) — Adam with **decoupled weight decay**. The default modern choice for training Transformers.

### The decoupled twist
Plain Adam + L2 regularization adds `λp` to the gradient *before* the moment updates, which means the L2 term is divided by `√v_hat` — different weights get different effective decay rates. AdamW applies decay **directly to the parameter** after the Adam update, so every weight gets the same effective decay:

$$p \leftarrow p \cdot (1 - \text{lr} \cdot \lambda)$$
$$p \leftarrow p - \text{lr} \cdot \hat{m} / (\sqrt{\hat{v}} + \varepsilon)$$

### Killer test (the interview line)
With `weight_decay > 0` and **zero gradients**, AdamW still shrinks the params (decoupled WD acts on `p` regardless of `g`). Plain Adam + L2 would do nothing in that case.


### Signature
```python
class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01): ...
    def zero_grad(self): ...
    def step(self): ...
```

### Rules
- Do **NOT** use `torch.optim.AdamW` or `torch.optim.Adam`
- Apply decoupled WD: `p *= (1 - lr * weight_decay)` to the param (NOT the gradient)
- Then Adam: m, v EMAs with bias correction, then `p -= lr * m_hat / (sqrt(v_hat) + eps)`
- Initialize `m, v` to zeros and step counter `t = 0`; increment `t` at the start of each step
- Skip params with `p.grad is None` (do NOT apply WD to them either — matches torch.optim.AdamW)
- Wrap `step()` in `@torch.no_grad()`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        pass  # store hyperparams; init m=[zeros], v=[zeros], t=0
    
    def zero_grad(self):
        pass
    
    def step(self):
        pass  # t += 1; for each (p, m, v) with p.grad is not None:
              #   decoupled WD: p *= (1 - lr*wd)
              #   m = β1*m + (1-β1)*g; v = β2*v + (1-β2)*g²
              #   m_hat = m / (1 - β1^t); v_hat = v / (1 - β2^t)
              #   p -= lr * m_hat / (sqrt(v_hat) + eps)

In [ ]:
import torch
torch.manual_seed(0)
p = torch.randn(5, requires_grad=True)
opt = MyAdamW([p], lr=0.01, weight_decay=0.1)
for step in range(3):
    loss = (p ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'step {step}: |p| = {p.norm().item():.4f}')

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("adamw")